In [ ]:
%pip install pandas
%pip install torch --index-url https://download.pytorch.org/whl/cu121
%pip install transformers
%pip install scikit-learn

In [ ]:
# Importing the libraries
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    TrainingArguments, 
    Trainer, 
    DataCollatorWithPadding
)
import torch
from torch.utils.data import Dataset
import gc

In [ ]:
# Load the local IMDB dataset
df = pd.read_csv('IMDB Dataset.csv')
df.head(5)

In [ ]:
# Create train and test splits (80-20 split)
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

In [ ]:
class IMDBDatasetForTrainer(Dataset):
    def __init__(self, df, tokenizer, max_length=128):
        self.reviews = df['review'].values
        self.sentiments = df['sentiment'].values
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.reviews)

    def __getitem__(self, idx):
        review = str(self.reviews[idx])
        label = 1 if self.sentiments[idx] == 'positive' else 0

        # Tokenize the review
        encoding = self.tokenizer(
            review,
            truncation=True,
            max_length=self.max_length,
            padding='max_length',
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'][0],
            'attention_mask': encoding['attention_mask'][0],
            'labels': torch.tensor(label, dtype=torch.long)
        }

In [ ]:
# Define metrics computation function
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return {"accuracy": (predictions == labels).mean()}

In [ ]:
# Initialize tokenizer and create datasets
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
train_dataset = IMDBDatasetForTrainer(train_df, tokenizer)
test_dataset = IMDBDatasetForTrainer(test_df, tokenizer)

# Initialize model
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", 
    num_labels=2
)

# Freeze base model parameters (optional)
for param in model.distilbert.parameters():
    param.requires_grad = False

model.classifier

# Move model to GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

In [ ]:
# Define base directory and model name
BASE_DIR = "./data"
MODEL_NAME = "sentiment_analysis"

# Create output directory
output_dir = os.path.join(BASE_DIR, MODEL_NAME)
os.makedirs(output_dir, exist_ok=True)

# Initialize training arguments
training_args = TrainingArguments(
    output_dir=output_dir,
    learning_rate=2e-3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=1,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    fp16=True,
    gradient_accumulation_steps=8,
    optim="adamw_torch",
    gradient_checkpointing=True,
    dataloader_num_workers=0
)

In [ ]:
# Initialize trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)

# Enable garbage collection
gc.enable()

# Train the model
print("Starting training...")
trainer.train()

# Clean up memory
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# Evaluate the model
print("\nEvaluating model...")
eval_results = trainer.evaluate()
print(f"Evaluation Results: {eval_results}")

In [ ]:
def predict_sentiment(model, tokenizer, text, device='cuda' if torch.cuda.is_available() else 'cpu'):
    """
    Predict sentiment for a given text with confidence score
    
    Args:
        model: The trained model
        tokenizer: The tokenizer used for the model
        text: Text input for sentiment analysis
        device: Device to run the model on ('cuda' or 'cpu')
    
    Returns:
        dict: Dictionary containing sentiment prediction, probability, and input text
    """
    model.eval()
    # Tokenize the text
    encoded_text = tokenizer(
        text,
        max_length=512,
        truncation=True,
        padding='max_length',
        return_tensors='pt'
    )
    
    # Move to device
    input_ids = encoded_text['input_ids'].to(device)
    attention_mask = encoded_text['attention_mask'].to(device)
    
    with torch.no_grad():
        outputs = model(input_ids, attention_mask=attention_mask)
        probabilities = torch.nn.functional.softmax(outputs.logits, dim=1)
        prediction = torch.argmax(probabilities, dim=1)
        confidence_score = probabilities[0][prediction].item()
    
    result = {
        'text': text,
        'sentiment': 'positive' if prediction.item() == 1 else 'negative',
        'confidence': f"{confidence_score:.2%}"
    }
    
    return result

In [ ]:
# Example usage:
# Single prediction
text = "This movie was really great! I enjoyed every minute of it."
result = predict_sentiment(model, tokenizer, text)
print(f"Text: {result['text']}")
print(f"Sentiment: {result['sentiment']}")
print(f"Confidence: {result['confidence']}")

# Multiple predictions
texts = [
    "This movie was really great! I enjoyed every minute of it.",
    "I wouldn't recommend this movie to anyone. It was terrible."
]

results = []
for text in texts:
    result = predict_sentiment(model, tokenizer, text)
    results.append(result)

# Create DataFrame with results
predictions_df = pd.DataFrame(results)
print("\nMultiple Predictions:")
print(predictions_df)